# 🌐 ONYX-Nexus: Tek Tıkla Colab Mesh Ağı Kurulumu (One-Click Colab Mesh)
### 5-Düğümlü Dağıtık Ajan Kümesi • Ücretsiz LLM Havuzu • Cloudflare Tünel • SQLite WAL FTS5

Bu notebook, Google Colab üzerinde **5 bağımsız mikroservis düğümünü (Mesh Nodes)** tek tıkla arka planda başlatır, Cloudflare Tüneli kurarak genel erişime açar ve web arayüzü ile senkronize eder.

In [ ]:
#@title 🚀 1. Tek Tıkla Kurulum & Başlatma (One-Click Mesh Launch)
#@markdown Aşağıdaki seçenekleri belirleyip hücreyi çalıştırın:

REPO_URL = "https://github.com/furkanarslangraydomain-stack/ONYX-Nexus.git" #@param {type:"string"}
ENABLE_CLOUDFLARE_TUNNEL = True #@param {type:"boolean"}
AUTO_INSTALL_DEPS = True #@param {type:"boolean"}
FREE_LLM_POOL_SIZE = 12 #@param {type:"integer"}

import os, sys, time, subprocess, shutil

print("=" * 75)
print("🚀 ONYX-NEXUS 5-NODE COLAB MESH ORCHESTRATOR BAŞLATILIYOR...")
print("=" * 75)

# 1. Depo Klonlama ve Senkronizasyon
TARGET_DIR = "/content/ONYX-Nexus" if os.path.exists("/content") else os.path.expanduser("~/ONYX-Nexus")
if not os.path.exists(TARGET_DIR):
    print(f"[*] Depo klonlanıyor: {REPO_URL} -> {TARGET_DIR}")
    subprocess.run(["git", "clone", "--depth=1", REPO_URL, TARGET_DIR], check=True)
else:
    print(f"[*] Depo mevcut, güncelleniyor: {TARGET_DIR}")
    subprocess.run(["git", "-C", TARGET_DIR, "pull"], check=False)

os.chdir(TARGET_DIR)

# 2. Bağımlılıkların Kurulumu
if AUTO_INSTALL_DEPS:
    print("[*] Temel mikroservis bağımlılıkları kuruluyor...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn", "httpx", "pydantic", "psutil", "python-dotenv", "pycloudflared"], check=True)

# 3. 5-Node Mesh Mikroservis Düğümlerini Tanımla ve Başlat
NODES = [
    {"id": 1, "name": "Master Orchestrator", "port": 8000, "role": "Orchestrator"},
    {"id": 2, "name": "Polyglot Compiler Sandbox", "port": 8001, "role": "Compiler Sandbox"},
    {"id": 3, "name": "Consensus Swarm & Deep Research", "port": 8002, "role": "Consensus Engine"},
    {"id": 4, "name": "3D Render Studio Engine", "port": 8003, "role": "3D Studio Engine"},
    {"id": 5, "name": "Distributed Vector DB & FTS5 Hub", "port": 8004, "role": "Memory Hub"}
]

print("\n[*] 5 Düğümlü Mesh Mikroservisleri Arka Planda Başlatılıyor...")
procs = []
for node in NODES:
    env = os.environ.copy()
    env["NODE_ID"] = str(node["id"])
    env["PORT"] = str(node["port"])
    env["HOST"] = "0.0.0.0"
    
    log_file = open(f"node_{node['port']}.log", "w")
    p = subprocess.Popen([sys.executable, "main.py"], env=env, stdout=log_file, stderr=subprocess.STDOUT)
    procs.append(p)
    print(f"  ✓ Düğüm {node['id']}: {node['name']} (Port: {node['port']}) [PID: {p.pid}] aktif.")
    time.sleep(1.2)

# 4. Cloudflare Tüneli Kurulumu
public_url = None
if ENABLE_CLOUDFLARE_TUNNEL:
    print("\n[*] Cloudflare Tüneli başlatılıyor (Port 8000 dış dünyaya açılıyor)...")
    try:
        from pycloudflared import try_cloudflare
        tunnel = try_cloudflare(port=8000)
        public_url = tunnel.tunnel_url
        print(f"\n🌟 [CLOUDFLARE PUBLIC TUNNEL AKTİF] -> {public_url}")
        print(f"Bu adresi ONYX-Nexus web arayüzündeki API_BASE_URL alanına yapıştırabilirsiniz!")
    except Exception as e:
        print(f"[!] Cloudflare tünel uyarısı: {e}. Yerel port 8000 açık.")

print("\n" + "=" * 75)
print("🎉 ONYX-NEXUS 5-NODE COLAB MESH AĞI TAM KAPASİTE ÇALIŞIYOR!")
print("=" * 75)
print(f"- Master URL: {public_url or 'http://127.0.0.1:8000'}")
print(f"- OpenAPI Dokümantasyonu: {public_url or 'http://127.0.0.1:8000'}/docs")
print(f"- Mesh Durumu: {public_url or 'http://127.0.0.1:8000'}/api/mesh/nodes")
print("-\nOturumun kapanmaması için arka plan döngüsü sürdürülüyor...")

# Canlı Tutma ve İzleme Döngüsü
try:
    while True:
        time.sleep(30)
except KeyboardInterrupt:
    print("\n[!] Mesh düğümleri kapatılıyor...")
    for p in procs:
        p.terminate()
    print("[*] Tamamlandı.")
